# Bài 25: Ghép ảnh và Tạo ảnh khảm (Image Stitching and Mosaicking)

Bài này kết hợp phát hiện và so khớp đặc trưng (Bài 20), khớp bền vững (Bài 23) và ước lượng phép đồng nhất (Bài 24) thành một quy trình hoàn chỉnh ghép hai bức ảnh chồng lấn thành một ảnh toàn cảnh (panorama) liền mạch duy nhất.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['clemson00.jpg', 'clemson01.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Quy trình, nhìn tổng quát

1. **Phát hiện và so khớp đặc trưng** giữa hai bức ảnh (Bài 20: SIFT + kiểm tra tỉ số).
2. **Ước lượng một phép đồng nhất** liên hệ mặt phẳng của ảnh này với ảnh kia, một cách bền vững (phép đồng nhất ở Bài 24, RANSAC ở Bài 23).
3. **Biến dạng** một ảnh vào hệ tọa độ của ảnh kia (Bài 9: `cv2.warpPerspective`).
4. **Ghép và trộn** hai ảnh lên một khung vẽ (canvas), làm mờ chuyển tiếp (feathering) qua vùng chồng lấn để đường nối không còn nhìn thấy.

## Hai bức ảnh chồng lấn

Hai bức ảnh chụp cùng một tòa nhà gạch, từ hai vị trí hơi khác nhau với phần chồng lấn đáng kể.

In [ ]:
view1 = cv2.imread('../img/clemson00.jpg')
view2 = cv2.imread('../img/clemson01.jpg')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(view1, cv2.COLOR_BGR2RGB))
axes[0].set_title('View 1')
axes[1].imshow(cv2.cvtColor(view2, cv2.COLOR_BGR2RGB))
axes[1].set_title('View 2')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Nguồn ảnh: Stan Birchfield</p>

## Bước 1: so khớp đặc trưng

In [ ]:
gray1 = cv2.cvtColor(view1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(view2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

bf = cv2.BFMatcher()
raw_matches = bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

print(f'keypoints: {len(kp1)} (view 1), {len(kp2)} (view 2)')
print(f'good matches after ratio test: {len(good_matches)}')

match_vis = cv2.drawMatches(view1, kp1, view2, kp2, good_matches[:60], None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(11, 4))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.title('Feature matches in the overlap region (first 60 shown)')
plt.axis('off')
plt.show()

## Bước 2: phép đồng nhất bền vững

Ta giải ra phép đồng nhất ánh xạ các điểm trong góc nhìn 2 vào hệ tọa độ của góc nhìn 1, để biến dạng góc nhìn 2 qua nó sẽ đặt nó vào đúng vị trí trên một khung vẽ chung.

In [ ]:
pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches])
pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches])

H, inlier_mask = cv2.findHomography(pts2, pts1, cv2.RANSAC, 3.0)

print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')
print('recovered homography:')
print(np.round(H, 4))

## Bước 3-4: biến dạng, ghép và trộn

Trước tiên ta xác định khung vẽ đầu ra cần lớn đến đâu, bằng cách ánh xạ các góc của cả hai ảnh qua `H` (Bài 24) và lấy hộp bao &mdash; khác với trường hợp tổng hợp, các bức ảnh này không phải là các vùng cắt thẳng trục đơn giản của một khung chung, nên cả kích thước khung vẽ lẫn độ lệch của góc nhìn 1 bên trong nó đều phải được tính toán thay vì giả định. Ta biến dạng góc nhìn 2 lên khung vẽ đó, rồi trộn vùng chồng lấn bằng một phép **làm mờ chuyển tiếp (feather)** đơn giản: một dốc alpha tuyến tính từ "hoàn toàn góc nhìn 1" đến "hoàn toàn góc nhìn 2" qua vùng chồng lấn, cùng kiểu trộn tổng có trọng số như `cv2.addWeighted` ở Bài 2, chỉ là với trọng số thay đổi theo không gian thay vì hằng số.

In [ ]:
h1, w1 = view1.shape[:2]
h2, w2 = view2.shape[:2]
corners1 = np.float32([[0, 0], [w1, 0], [w1, h1], [0, h1]]).reshape(-1, 1, 2)
corners2 = np.float32([[0, 0], [w2, 0], [w2, h2], [0, h2]]).reshape(-1, 1, 2)
warped_corners2 = cv2.perspectiveTransform(corners2, H)
all_corners = np.concatenate([corners1, warped_corners2], axis=0)

x_min, y_min = np.floor(all_corners.min(axis=0).ravel()).astype(int)
x_max, y_max = np.ceil(all_corners.max(axis=0).ravel()).astype(int)
canvas_w, canvas_h = x_max - x_min, y_max - y_min

translation = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float64)
canvas1 = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
canvas1[-y_min:-y_min + h1, -x_min:-x_min + w1] = view1
warped2 = cv2.warpPerspective(view2, translation @ H, (canvas_w, canvas_h))

has1 = canvas1.sum(axis=2) > 0
has2 = warped2.sum(axis=2) > 0
overlap = has1 & has2

overlap_cols = np.where(overlap.any(axis=0))[0]
x_start, x_end = overlap_cols.min(), overlap_cols.max()
ramp = np.clip((np.arange(canvas_w) - x_start) / (x_end - x_start + 1e-6), 0, 1)

alpha = np.zeros((canvas_h, canvas_w), dtype=np.float32)
alpha[overlap] = np.broadcast_to(ramp, (canvas_h, canvas_w))[overlap]

stitched = canvas1.astype(np.float32) * (1 - alpha[..., None]) + warped2.astype(np.float32) * alpha[..., None]
stitched[has1 & ~has2] = canvas1[has1 & ~has2]   # regions covered only by view 1
stitched[has2 & ~has1] = warped2[has2 & ~has1]   # regions covered only by view 2
stitched = stitched.astype(np.uint8)

plt.figure(figsize=(10, 4))
plt.imshow(cv2.cvtColor(stitched, cv2.COLOR_BGR2RGB))
plt.title('Stitched panorama')
plt.axis('off')
plt.show()

Ảnh động này chạy toàn bộ quy trình ở trên trên cùng hai bức ảnh: các cặp khớp tích lũy dần qua phép kiểm tra tỉ số, phép đồng nhất khớp bằng RANSAC biến dạng góc nhìn 2 vào hệ tọa độ của góc nhìn 1, và phép trộn làm mờ chuyển tiếp hiện dần qua vùng chồng lấn để tạo ra ảnh toàn cảnh cuối cùng.

<img src="../animations/stitching_pipeline_demo.gif" alt="Ảnh động quy trình ghép ảnh: các cặp khớp SIFT tích lũy dần, một phép đồng nhất RANSAC biến dạng góc nhìn 2 vào hệ tọa độ của góc nhìn 1, rồi phép trộn làm mờ chuyển tiếp hiện dần qua vùng chồng lấn để tạo ra ảnh toàn cảnh" style="display: block; margin: 0 auto; max-width: 680px;">

Mái nhà (phía trên, chỉ thấy đầy đủ trong góc nhìn 2) và bụi cây (phía dưới, chỉ thấy đầy đủ trong góc nhìn 1) đều xuất hiện trong ảnh khảm &mdash; xác nhận rằng ảnh ghép thực sự lấy từ cả hai ảnh nguồn.

### Phép khớp tốt đến đâu?

Với ảnh chụp thật thì không có dữ liệu chuẩn tổng hợp để so sánh, nhưng ta không cần đến nó: bản thân các điểm trong của RANSAC đã cho một thước đo trực tiếp về độ khớp hình học. Với mỗi cặp tương ứng điểm trong, chiếu điểm từ góc nhìn 2 qua `H` và đo khoảng cách tới điểm khớp của nó trong góc nhìn 1 &mdash; gọi là **sai số chiếu lại (reprojection error)**. Sai số chiếu lại nhỏ, tập trung chặt chẽ nghĩa là `H` giải thích tốt hình học của các điểm trong.

In [ ]:
mask = inlier_mask.ravel().astype(bool)
pts2_h = np.hstack([pts2[mask], np.ones((mask.sum(), 1))])
proj = (H @ pts2_h.T).T
proj = proj[:, :2] / proj[:, 2:3]
reproj_error = np.linalg.norm(proj - pts1[mask], axis=1)

print(f'mean reprojection error: {reproj_error.mean():.3f} px')
print(f'max reprojection error:  {reproj_error.max():.3f} px')

plt.figure(figsize=(6, 3.5))
plt.hist(reproj_error, bins=20)
plt.xlabel('reprojection error (px)')
plt.ylabel('inlier count')
plt.title('Reprojection error of RANSAC inliers')
plt.tight_layout()
plt.show()

Sai số chiếu lại ở cấp dưới điểm ảnh, thấp hơn nhiều so với ngưỡng điểm trong RANSAC là 3 px, xác nhận `H` là một phép khớp hình học tốt &mdash; nhất quán với đường nối sạch sẽ thấy được trong ảnh toàn cảnh đã ghép ở trên.

## Trong thực tế: `cv2.Stitcher`

OpenCV gói toàn bộ quy trình này (cùng với phép trộn bền vững hơn, bù phơi sáng và hỗ trợ nhiều ảnh cùng lúc, sắp xếp theo bất kỳ cấu hình nào) sau một lệnh gọi cấp cao duy nhất. Chế độ `PANORAMA` mặc định của nó giả định các ảnh đến từ một camera xoay quanh tâm quang học của nó, và biến dạng mỗi ảnh lên một mặt cầu trước khi ghép &mdash; tuyệt vời cho ảnh toàn cảnh rộng, nhưng nó làm cong các đường thẳng khi cảnh bị chi phối bởi cấu trúc phẳng, thẳng góc (như mặt tiền tòa nhà) và camera tịnh tiến thay vì chỉ xoay. Chế độ `SCANS` thay vào đó ghép trực tiếp bằng các phép đồng nhất phẳng, khớp với cách tiếp cận dùng ở trên.

In [ ]:
stitcher = cv2.Stitcher_create(cv2.Stitcher_SCANS)
status, panorama = stitcher.stitch([view1, view2])

print('status:', 'OK' if status == cv2.Stitcher_OK else f'failed ({status})')
if status == cv2.Stitcher_OK:
    plt.figure(figsize=(10, 4))
    plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
    plt.title('cv2.Stitcher result (SCANS mode)')
    plt.axis('off')
    plt.show()

### Bài tập

1. Giảm phần chồng lấn giữa hai góc nhìn (ví dụ cắt `view2` chỉ còn một phần tư bên phải trước khi so khớp). Đến mức nào thì so khớp SIFT tìm được quá ít cặp khớp tốt để `findHomography` cho ra kết quả đáng tin cậy?
2. Thay phép làm mờ chuyển tiếp tuyến tính bằng một ngưỡng cắt cứng (không trộn: chỉ chọn ảnh nào phủ mỗi điểm ảnh, chia đôi vùng chồng lấn ở giữa) và so sánh chất lượng đường nối thấy được với phiên bản làm mờ chuyển tiếp.
3. `H` ở đây là một phép đồng nhất tổng quát nhẹ, không phải phép tịnh tiến thuần túy, vì hai bức ảnh được chụp từ hai vị trí hơi khác nhau thay vì chỉ lia ngang. In tỉ số giữa hàng cuối của `H` với `[0, 0, 1]` như một thước đo thô về lượng méo phối cảnh mà nó nắm bắt, và so sánh với kết quả khi ép buộc một phép khớp affine (`cv2.estimateAffinePartial2D`) thay vào đó &mdash; phép xấp xỉ affine ghép ảnh tốt đến đâu so với phép đồng nhất đầy đủ?